# Chronological run-level split design

This notebook analyzes complete runs and compares safe train/validation/test assignments. It proposes strategies only; it does not create final split datasets.

### 1. Define paths and fingerprint the input

**What the cell does:** Imports libraries, defines repository-relative paths, and records the feature dataset's SHA-256 checksum.  
**Why it is important:** Split analysis must be reproducible and must not modify the modeling table.  
**What to understand:** The fingerprint identifies the exact input and will be verified again after report creation.

In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
INPUT_PATH = PROJECT_ROOT / "data" / "interim" / "feature_dataset.csv"
RUN_CANDIDATES_PATH = PROJECT_ROOT / "reports" / "run_split_candidates.csv"
STRATEGY_COMPARISON_PATH = PROJECT_ROOT / "reports" / "split_strategy_comparison.csv"

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"Feature dataset not found: {INPUT_PATH}")
input_hash_before = sha256_file(INPUT_PATH)
print(f"Input: {INPUT_PATH}")
print(f"Input SHA-256 before analysis: {input_hash_before}")

Input: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/interim/feature_dataset.csv
Input SHA-256 before analysis: 29f6dba2b376bd26a586b6ad332a2db5bf043e5cad64f92f658b34fc85c9dc67


### 2. Load and validate a chronological analysis copy

**What the cell does:** Loads the feature dataset, validates sequence/target columns, safely parses timestamps, and stably sorts by machine, timestamp, run, and segment.  
**Why it is important:** Chronological split design depends on reliable run timing and must never be based on random row order.  
**What to understand:** All rows belong to a valid machine/run/segment sequence and the input itself remains unchanged.

In [2]:
feature_input = pd.read_csv(INPUT_PATH)
required_columns = {"machine_id", "run_id", "segment_id", "timestamp", "slowdown_in_5min"}
missing_columns = required_columns - set(feature_input.columns)
if missing_columns:
    raise KeyError(f"Required split-design columns are missing: {sorted(missing_columns)}")

analysis_data = feature_input.copy(deep=True)
analysis_data["_source_row"] = np.arange(len(analysis_data))
analysis_data["_timestamp_dt"] = pd.to_datetime(analysis_data["timestamp"], errors="coerce", utc=True)
if analysis_data["_timestamp_dt"].isna().any():
    raise ValueError(f"Invalid timestamps found: {analysis_data['_timestamp_dt'].isna().sum()}")
if analysis_data[list(required_columns)].isna().any().any():
    raise ValueError("Required split-design columns contain missing values.")

analysis_data = analysis_data.sort_values(
    ["machine_id", "_timestamp_dt", "run_id", "segment_id", "_source_row"],
    kind="mergesort",
).copy()

print(f"Rows: {len(analysis_data):,}")
print(f"Machines: {analysis_data['machine_id'].nunique()}")
print(f"Runs: {analysis_data['run_id'].nunique()}")
print(f"Segments: {analysis_data['segment_id'].nunique()}")

Rows: 26,036
Machines: 3
Runs: 10
Segments: 33


### 3. Summarize every complete run

**What the cell does:** Creates one row per machine/run with time range, duration, rows, target counts/rate, segment count, and chronological rank within its machine.  
**Why it is important:** Runs—not rows—are the indivisible units of a leakage-safe split.  
**What to understand:** The displayed ten rows describe the size, label balance, and chronological position of every candidate run.

In [3]:
run_split_candidates = (
    analysis_data.groupby(["machine_id", "run_id"], dropna=False)
    .agg(
        start_timestamp=("_timestamp_dt", "min"),
        end_timestamp=("_timestamp_dt", "max"),
        total_rows=("_source_row", "size"),
        positive_rows=("slowdown_in_5min", lambda values: int(values.eq(1).sum())),
        negative_rows=("slowdown_in_5min", lambda values: int(values.eq(0).sum())),
        number_of_segments=("segment_id", "nunique"),
    )
    .reset_index()
)
run_split_candidates["duration_minutes"] = (
    run_split_candidates["end_timestamp"] - run_split_candidates["start_timestamp"]
).dt.total_seconds() / 60
run_split_candidates["positive_rate"] = (
    run_split_candidates["positive_rows"] / run_split_candidates["total_rows"] * 100
).round(4)
run_split_candidates = run_split_candidates.sort_values(
    ["machine_id", "start_timestamp", "run_id"], kind="mergesort"
).reset_index(drop=True)
run_split_candidates["chronological_order_within_machine"] = (
    run_split_candidates.groupby("machine_id").cumcount() + 1
)
run_split_candidates["runs_on_machine"] = run_split_candidates.groupby("machine_id")["run_id"].transform("size")

ordered_columns = [
    "machine_id", "run_id", "start_timestamp", "end_timestamp", "duration_minutes",
    "total_rows", "positive_rows", "negative_rows", "positive_rate",
    "number_of_segments", "chronological_order_within_machine", "runs_on_machine"
]
run_split_candidates = run_split_candidates[ordered_columns]
assert len(run_split_candidates) == 10 and run_split_candidates["run_id"].is_unique
display(run_split_candidates)

,machine_id,run_id,start_timestamp,end_timestamp,duration_minutes,total_rows,positive_rows,negative_rows,positive_rate,number_of_segments,chronological_order_within_machine,runs_on_machine
0,0890dcc046c079acc4de4202,89cdc34b-e02b-43b4-9284-144276df508a,2026-07-23 15:00:46.964000+00:00,2026-07-23 19:58:08.855000+00:00,297.364850,8484,256,8228,3.0174,2,1,2
1,0890dcc046c079acc4de4202,6835f125-a038-4092-beff-5107ae998b39,2026-07-25 14:45:59.803000+00:00,2026-07-25 16:40:54.978000+00:00,114.919583,3449,344,3105,9.9739,1,2,2
2,7232bc533c21ce408d45d473,d88b15dd-1915-43ca-90ef-69f31ff4d9c1,2026-07-28 10:40:53.862000+00:00,2026-07-28 13:05:14.375000+00:00,144.341883,1165,311,854,26.6953,10,1,4
3,7232bc533c21ce408d45d473,ec61755d-b5be-42ac-875d-3123e92add7a,2026-07-28 15:15:18.443000+00:00,2026-07-28 17:44:18.127000+00:00,148.994733,1302,62,1240,4.7619,7,2,4
4,7232bc533c21ce408d45d473,fac82c2e-545a-402a-80f8-3d1fccb72c68,2026-07-29 07:39:50.103000+00:00,2026-07-29 09:35:10.620000+00:00,115.341950,1611,710,901,44.0720,7,3,4
5,7232bc533c21ce408d45d473,19127a70-e60c-4b47-b3e6-71e89175c174,2026-07-29 13:35:20.985000+00:00,2026-07-29 13:40:23.654000+00:00,5.044483,119,105,14,88.2353,1,4,4
6,a0f8c86097e55fbfa506d057,2e9f4457-2a7f-4647-87ed-b86ac6343d33,2026-07-27 12:17:19.031000+00:00,2026-07-27 12:29:16.187000+00:00,11.952600,182,0,182,0.0000,1,1,4
7,a0f8c86097e55fbfa506d057,f24e9c1a-f11e-405c-b663-45fa6e25c405,2026-07-27 13:16:45.456000+00:00,2026-07-27 15:27:24.379000+00:00,130.648717,3860,453,3407,11.7358,1,2,4
8,a0f8c86097e55fbfa506d057,373070d1-ba59-4244-88ab-2d44a21f4983,2026-07-27 22:57:32.661000+00:00,2026-07-27 23:06:14.948000+00:00,8.704783,261,0,261,0.0000,1,3,4
9,a0f8c86097e55fbfa506d057,70577f8e-1430-4489-8602-2096521ab84e,2026-07-28 17:22:42.771000+00:00,2026-07-28 21:12:01.059000+00:00,229.304800,5603,1402,4201,25.0223,2,4,4


### 4. Flag risky run characteristics and limited machine coverage

**What the cell does:** Flags zero-positive runs, rates of at least 40%, runs below 500 rows, and machines with fewer than three runs.  
**Why it is important:** Tiny, single-class, or extreme runs can produce unstable validation/test estimates, while sparse machine history limits three-way coverage.  
**What to understand:** Risk flags do not remove runs; they explain why a proposed holdout may be unreliable or unrepresentative.

In [4]:
VERY_HIGH_POSITIVE_RATE = 40.0
VERY_FEW_ROWS = 500
LIMITED_MACHINE_RUNS = 3

run_split_candidates["zero_positive_labels"] = run_split_candidates["positive_rows"].eq(0)
run_split_candidates["very_high_positive_rate"] = run_split_candidates["positive_rate"].ge(VERY_HIGH_POSITIVE_RATE)
run_split_candidates["very_few_rows"] = run_split_candidates["total_rows"].lt(VERY_FEW_ROWS)
run_split_candidates["limited_machine_run_coverage"] = run_split_candidates["runs_on_machine"].lt(LIMITED_MACHINE_RUNS)
run_split_candidates["risky_for_validation_or_test"] = run_split_candidates[[
    "zero_positive_labels", "very_high_positive_rate", "very_few_rows"
]].any(axis=1)
run_split_candidates["risk_reasons"] = run_split_candidates.apply(
    lambda row: ";".join([
        reason for condition, reason in [
            (row["zero_positive_labels"], "zero_positive_labels"),
            (row["very_high_positive_rate"], "very_high_positive_rate"),
            (row["very_few_rows"], "very_few_rows"),
            (row["limited_machine_run_coverage"], "limited_machine_run_coverage"),
        ] if condition
    ]), axis=1
)

print("Runs requiring special caution:")
display(run_split_candidates.loc[
    run_split_candidates["risky_for_validation_or_test"] | run_split_candidates["limited_machine_run_coverage"],
    ["machine_id", "run_id", "total_rows", "positive_rate", "risk_reasons"]
])

Runs requiring special caution:


,machine_id,run_id,total_rows,positive_rate,risk_reasons
0,0890dcc046c079acc4de4202,89cdc34b-e02b-43b4-9284-144276df508a,8484,3.0174,limited_machine_run_coverage
1,0890dcc046c079acc4de4202,6835f125-a038-4092-beff-5107ae998b39,3449,9.9739,limited_machine_run_coverage
4,7232bc533c21ce408d45d473,fac82c2e-545a-402a-80f8-3d1fccb72c68,1611,44.0720,very_high_positive_rate
5,7232bc533c21ce408d45d473,19127a70-e60c-4b47-b3e6-71e89175c174,119,88.2353,very_high_positive_rate;very_few_rows
6,a0f8c86097e55fbfa506d057,2e9f4457-2a7f-4647-87ed-b86ac6343d33,182,0.0000,zero_positive_labels;very_few_rows
8,a0f8c86097e55fbfa506d057,373070d1-ba59-4244-88ab-2d44a21f4983,261,0.0000,zero_positive_labels;very_few_rows


### 5. Define four complete-run chronological strategies

**What the cell does:** Assigns every run exactly once using four transparent approaches: latest-run holdout per machine, global time blocks, a larger multi-run future test, and broader validation-machine coverage.  
**Why it is important:** Multiple valid designs expose tradeoffs between proportions, machine coverage, and realistic future evaluation.  
**What to understand:** No row is randomized and no run is divided; the two-run machine cannot appear in all three splits.

In [5]:
strategy_descriptions = {
    "per_machine_latest_run_test": "Per machine: latest run test, preceding run validation when available, all earlier runs train; a two-run machine contributes train and test.",
    "global_chronological_blocks": "All runs ordered globally by start time: earliest 4 train, next 3 validation, latest 3 test.",
    "multi_run_future_test": "Per four-run machine: earliest train, second validation, latest two test; a two-run machine contributes train and test.",
    "validation_machine_coverage": "Like latest-run holdout, but the two-run machine's latest run is validation so validation covers all machines.",
}

assignments = {}

latest_holdout = {}
for row in run_split_candidates.itertuples(index=False):
    rank = row.chronological_order_within_machine
    count = row.runs_on_machine
    if count == 2:
        split = "train" if rank == 1 else "test"
    elif rank == count:
        split = "test"
    elif rank == count - 1:
        split = "validation"
    else:
        split = "train"
    latest_holdout[row.run_id] = split
assignments["per_machine_latest_run_test"] = latest_holdout

global_runs = run_split_candidates.sort_values(["start_timestamp", "machine_id", "run_id"]).reset_index(drop=True)
assignments["global_chronological_blocks"] = {
    row.run_id: ("train" if position < 4 else "validation" if position < 7 else "test")
    for position, row in enumerate(global_runs.itertuples(index=False))
}

multi_future = {}
for row in run_split_candidates.itertuples(index=False):
    rank = row.chronological_order_within_machine
    count = row.runs_on_machine
    if count == 2:
        split = "train" if rank == 1 else "test"
    else:
        split = "train" if rank == 1 else "validation" if rank == 2 else "test"
    multi_future[row.run_id] = split
assignments["multi_run_future_test"] = multi_future

validation_coverage = {}
for row in run_split_candidates.itertuples(index=False):
    rank = row.chronological_order_within_machine
    count = row.runs_on_machine
    if count == 2:
        split = "train" if rank == 1 else "validation"
    elif rank == count:
        split = "test"
    elif rank == count - 1:
        split = "validation"
    else:
        split = "train"
    validation_coverage[row.run_id] = split
assignments["validation_machine_coverage"] = validation_coverage

all_run_ids = set(run_split_candidates["run_id"])
for strategy_name, mapping in assignments.items():
    assert set(mapping) == all_run_ids
    assert set(mapping.values()) == {"train", "validation", "test"}
    print(f"{strategy_name}:")
    for split_name in ["train", "validation", "test"]:
        print(f"  {split_name}: {[run_id for run_id, split in mapping.items() if split == split_name]}")

per_machine_latest_run_test:
  train: ['89cdc34b-e02b-43b4-9284-144276df508a', 'd88b15dd-1915-43ca-90ef-69f31ff4d9c1', 'ec61755d-b5be-42ac-875d-3123e92add7a', '2e9f4457-2a7f-4647-87ed-b86ac6343d33', 'f24e9c1a-f11e-405c-b663-45fa6e25c405']
  validation: ['fac82c2e-545a-402a-80f8-3d1fccb72c68', '373070d1-ba59-4244-88ab-2d44a21f4983']
  test: ['6835f125-a038-4092-beff-5107ae998b39', '19127a70-e60c-4b47-b3e6-71e89175c174', '70577f8e-1430-4489-8602-2096521ab84e']
global_chronological_blocks:
  train: ['89cdc34b-e02b-43b4-9284-144276df508a', '6835f125-a038-4092-beff-5107ae998b39', '2e9f4457-2a7f-4647-87ed-b86ac6343d33', 'f24e9c1a-f11e-405c-b663-45fa6e25c405']
  validation: ['373070d1-ba59-4244-88ab-2d44a21f4983', 'd88b15dd-1915-43ca-90ef-69f31ff4d9c1', 'ec61755d-b5be-42ac-875d-3123e92add7a']
  test: ['70577f8e-1430-4489-8602-2096521ab84e', 'fac82c2e-545a-402a-80f8-3d1fccb72c68', '19127a70-e60c-4b47-b3e6-71e89175c174']
multi_run_future_test:
  train: ['89cdc34b-e02b-43b4-9284-144276df508a', '

### 6. Evaluate split metrics, overlap, chronology, and realism

**What the cell does:** Calculates rows, proportions, classes, rates, machines, runs, and both-class checks for every split, then validates run exclusivity and nondecreasing split order inside each machine.  
**Why it is important:** A safe strategy needs causal ordering and realistic coverage—not merely similar class rates.  
**What to understand:** Strategy-level flags identify proportion problems, extreme test sets, missing machine coverage, or any leakage-causing overlap.

In [6]:
split_order = {"train": 0, "validation": 1, "test": 2}
total_rows = len(analysis_data)
strategy_rows = []
strategy_level = {}

for strategy_name, mapping in assignments.items():
    assignment_table = run_split_candidates[["machine_id", "run_id", "start_timestamp"]].copy()
    assignment_table["split"] = assignment_table["run_id"].map(mapping)
    run_overlap_detected = len(mapping) != len(set(mapping)) or set(mapping) != all_run_ids
    chronology_respected = True
    for _, machine_runs in assignment_table.sort_values(["machine_id", "start_timestamp"]).groupby("machine_id"):
        stages = machine_runs["split"].map(split_order).tolist()
        chronology_respected = chronology_respected and stages == sorted(stages)

    split_metrics = {}
    for split_name in ["train", "validation", "test"]:
        run_ids = [run_id for run_id, assigned_split in mapping.items() if assigned_split == split_name]
        split_data = analysis_data.loc[analysis_data["run_id"].isin(run_ids)]
        rows = int(len(split_data))
        positives = int(split_data["slowdown_in_5min"].eq(1).sum())
        negatives = int(split_data["slowdown_in_5min"].eq(0).sum())
        split_metrics[split_name] = {
            "rows": rows,
            "percentage": rows / total_rows * 100,
            "positives": positives,
            "negatives": negatives,
            "positive_rate": positives / rows * 100 if rows else np.nan,
            "machine_count": int(split_data["machine_id"].nunique()),
            "run_count": len(run_ids),
            "both_classes": positives > 0 and negatives > 0,
            "run_ids": run_ids,
        }

    reasonable_proportions = bool(
        45 <= split_metrics["train"]["percentage"] <= 75
        and 5 <= split_metrics["validation"]["percentage"] <= 25
        and 15 <= split_metrics["test"]["percentage"] <= 40
    )
    extreme_test_set = bool(
        split_metrics["test"]["rows"] < 1000
        or split_metrics["test"]["percentage"] < 10
        or split_metrics["test"]["percentage"] > 40
        or split_metrics["test"]["positive_rate"] < 1
        or split_metrics["test"]["positive_rate"] > 50
    )
    all_splits_both_classes = all(values["both_classes"] for values in split_metrics.values())
    coverage_score = (
        3 * split_metrics["test"]["machine_count"] / 3
        + 2 * split_metrics["validation"]["machine_count"] / 3
        + 2 * split_metrics["train"]["machine_count"] / 3
        + 2 * int(reasonable_proportions)
        + int(split_metrics["test"]["run_count"] >= 3)
        - 2 * int(extreme_test_set)
    )
    strategy_score = coverage_score if (not run_overlap_detected and chronology_respected and all_splits_both_classes) else -np.inf
    strategy_level[strategy_name] = {
        "run_overlap_detected": run_overlap_detected,
        "chronology_respected_per_machine": chronology_respected,
        "all_splits_have_both_classes": all_splits_both_classes,
        "reasonable_dataset_proportions": reasonable_proportions,
        "extreme_test_set_warning": extreme_test_set,
        "strategy_score": strategy_score,
    }
    for split_name, metrics in split_metrics.items():
        strategy_rows.append({
            "strategy_name": strategy_name,
            "strategy_description": strategy_descriptions[strategy_name],
            "split": split_name,
            "assigned_run_ids": json.dumps(metrics["run_ids"]),
            "row_count": metrics["rows"],
            "percentage_of_total_rows": round(metrics["percentage"], 4),
            "positive_rows": metrics["positives"],
            "negative_rows": metrics["negatives"],
            "positive_rate": round(metrics["positive_rate"], 4),
            "machine_count": metrics["machine_count"],
            "run_count": metrics["run_count"],
            "both_classes_exist": metrics["both_classes"],
            **strategy_level[strategy_name],
        })

split_strategy_comparison = pd.DataFrame(strategy_rows)
display(split_strategy_comparison)

,strategy_name,strategy_description,split,assigned_run_ids,row_count,percentage_of_total_rows,positive_rows,negative_rows,positive_rate,machine_count,run_count,both_classes_exist,run_overlap_detected,chronology_respected_per_machine,all_splits_have_both_classes,reasonable_dataset_proportions,extreme_test_set_warning,strategy_score
0,per_machine_latest_run_test,"Per machine: latest run test, preceding run va...",train,"[""89cdc34b-e02b-43b4-9284-144276df508a"", ""d88b...",14993,57.5857,1082,13911,7.2167,3,5,True,False,True,True,True,False,9.333333
1,per_machine_latest_run_test,"Per machine: latest run test, preceding run va...",validation,"[""fac82c2e-545a-402a-80f8-3d1fccb72c68"", ""3730...",1872,7.1900,710,1162,37.9274,2,2,True,False,True,True,True,False,9.333333
2,per_machine_latest_run_test,"Per machine: latest run test, preceding run va...",test,"[""6835f125-a038-4092-beff-5107ae998b39"", ""1912...",9171,35.2243,1851,7320,20.1832,3,3,True,False,True,True,True,False,9.333333
3,global_chronological_blocks,All runs ordered globally by start time: earli...,train,"[""89cdc34b-e02b-43b4-9284-144276df508a"", ""6835...",15975,61.3574,1053,14922,6.5915,2,4,True,False,True,True,True,False,7.666667
4,global_chronological_blocks,All runs ordered globally by start time: earli...,validation,"[""373070d1-ba59-4244-88ab-2d44a21f4983"", ""d88b...",2728,10.4778,373,2355,13.6730,2,3,True,False,True,True,True,False,7.666667
5,global_chronological_blocks,All runs ordered globally by start time: earli...,test,"[""70577f8e-1430-4489-8602-2096521ab84e"", ""fac8...",7333,28.1648,2217,5116,30.2332,2,3,True,False,True,True,True,False,7.666667
6,multi_run_future_test,"Per four-run machine: earliest train, second v...",train,"[""89cdc34b-e02b-43b4-9284-144276df508a"", ""d88b...",9831,37.7593,567,9264,5.7675,3,3,True,False,True,True,False,True,5.333333
7,multi_run_future_test,"Per four-run machine: earliest train, second v...",validation,"[""ec61755d-b5be-42ac-875d-3123e92add7a"", ""f24e...",5162,19.8264,515,4647,9.9768,2,2,True,False,True,True,False,True,5.333333
8,multi_run_future_test,"Per four-run machine: earliest train, second v...",test,"[""6835f125-a038-4092-beff-5107ae998b39"", ""fac8...",11043,42.4143,2561,8482,23.1912,3,5,True,False,True,True,False,True,5.333333
9,validation_machine_coverage,"Like latest-run holdout, but the two-run machi...",train,"[""89cdc34b-e02b-43b4-9284-144276df508a"", ""d88b...",14993,57.5857,1082,13911,7.2167,3,5,True,False,True,True,True,False,8.000000


### 7. Recommend the safest provisional strategy

**What the cell does:** Ranks only overlap-free, chronological, two-class strategies using test-machine coverage, validation/train coverage, proportions, and multiple future test runs.  
**Why it is important:** The recommendation should prioritize a realistic future test rather than artificially matching positive rates.  
**What to understand:** The selected strategy is a design proposal; its remaining limitations must be reviewed before materializing datasets.

In [7]:
strategy_ranking = pd.DataFrame(strategy_level).T.sort_values("strategy_score", ascending=False)
recommended_strategy = strategy_ranking.index[0]
split_strategy_comparison["recommended_strategy"] = split_strategy_comparison["strategy_name"].eq(recommended_strategy)
split_strategy_comparison["recommendation_status"] = np.where(
    split_strategy_comparison["recommended_strategy"],
    "proposal_requires_manual_approval",
    "alternative_not_selected",
)

print("Strategy-level ranking:")
display(strategy_ranking)
print(f"Recommended provisional strategy: {recommended_strategy}")
print(strategy_descriptions[recommended_strategy])
print("Reason: it preserves chronology and complete runs, retains both classes, and gives the future test set coverage of all three machines.")
print("Remaining limitation: validation excludes the two-run machine and is relatively small, so uncertainty must be reported.")

Strategy-level ranking:


,run_overlap_detected,chronology_respected_per_machine,all_splits_have_both_classes,reasonable_dataset_proportions,extreme_test_set_warning,strategy_score
per_machine_latest_run_test,False,True,True,True,False,9.333333
validation_machine_coverage,False,True,True,True,False,8.0
global_chronological_blocks,False,True,True,True,False,7.666667
multi_run_future_test,False,True,True,False,True,5.333333


Recommended provisional strategy: per_machine_latest_run_test
Per machine: latest run test, preceding run validation when available, all earlier runs train; a two-run machine contributes train and test.
Reason: it preserves chronology and complete runs, retains both classes, and gives the future test set coverage of all three machines.
Remaining limitation: validation excludes the two-run machine and is relatively small, so uncertainty must be reported.


### 8. Save reports and verify that no final split was created

**What the cell does:** Saves the run candidate and strategy comparison reports, validates them, and confirms the feature dataset checksum is unchanged.  
**Why it is important:** This stage must stop at design analysis—without producing train, validation, or test datasets.  
**What to understand:** Successful output provides reviewable assignments and risks while leaving the input intact and awaiting manual strategy approval.

In [8]:
RUN_CANDIDATES_PATH.parent.mkdir(parents=True, exist_ok=True)
run_split_candidates.to_csv(RUN_CANDIDATES_PATH, index=False)
split_strategy_comparison.to_csv(STRATEGY_COMPARISON_PATH, index=False)

saved_runs = pd.read_csv(RUN_CANDIDATES_PATH)
saved_strategies = pd.read_csv(STRATEGY_COMPARISON_PATH)
assert len(saved_runs) == 10 and saved_runs["run_id"].is_unique
assert len(saved_strategies) == len(assignments) * 3
assert saved_strategies.groupby("strategy_name")["split"].nunique().eq(3).all()
assert not saved_strategies["run_overlap_detected"].any()
assert saved_strategies["chronology_respected_per_machine"].all()

input_hash_after = sha256_file(INPUT_PATH)
input_unchanged = input_hash_before == input_hash_after
assert input_unchanged, "The feature dataset changed during split design."

print("FINAL SPLIT-DESIGN REPORT")
print(f"Runs analyzed: {len(run_split_candidates)}")
print(f"Zero-positive runs: {run_split_candidates.loc[run_split_candidates['zero_positive_labels'], 'run_id'].tolist()}")
print(f"Very-high-positive runs (≥{VERY_HIGH_POSITIVE_RATE:g}%): {run_split_candidates.loc[run_split_candidates['very_high_positive_rate'], 'run_id'].tolist()}")
print(f"Very-small runs (<{VERY_FEW_ROWS} rows): {run_split_candidates.loc[run_split_candidates['very_few_rows'], 'run_id'].tolist()}")
print(f"Machines with limited run coverage: {run_split_candidates.loc[run_split_candidates['limited_machine_run_coverage'], 'machine_id'].unique().tolist()}")
print(f"Strategies evaluated: {list(assignments)}")
print(f"Recommended provisional strategy: {recommended_strategy}")
print(f"Input feature dataset remained unchanged: {input_unchanged}")
print("No final train, validation, or test files were created.")
print("No random splitting, normalization, imputation, balancing, or model training was performed.")
print(f"Created: {RUN_CANDIDATES_PATH}")
print(f"Created: {STRATEGY_COMPARISON_PATH}")

FINAL SPLIT-DESIGN REPORT
Runs analyzed: 10
Zero-positive runs: ['2e9f4457-2a7f-4647-87ed-b86ac6343d33', '373070d1-ba59-4244-88ab-2d44a21f4983']
Very-high-positive runs (≥40%): ['fac82c2e-545a-402a-80f8-3d1fccb72c68', '19127a70-e60c-4b47-b3e6-71e89175c174']
Very-small runs (<500 rows): ['19127a70-e60c-4b47-b3e6-71e89175c174', '2e9f4457-2a7f-4647-87ed-b86ac6343d33', '373070d1-ba59-4244-88ab-2d44a21f4983']
Machines with limited run coverage: ['0890dcc046c079acc4de4202']
Strategies evaluated: ['per_machine_latest_run_test', 'global_chronological_blocks', 'multi_run_future_test', 'validation_machine_coverage']
Recommended provisional strategy: per_machine_latest_run_test
Input feature dataset remained unchanged: True
No final train, validation, or test files were created.
No random splitting, normalization, imputation, balancing, or model training was performed.
Created: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/run_split_candidates.csv
Created: /Users/fatimazahranamaoui